In [ ]:
# ============================================================
# CREDORA FINANCE — Credit Risk & Loan Approval Analytics
# Notebook 03: Modeling (model comparison, champion selection, SHAP)
# Split from the combined script: risk scoring moved to 04_risk_scoring
# ============================================================

# ============================================================
# CREDORA FINANCE — Credit Risk & Loan Approval Analytics
# Notebook 03: Advanced Models, Tuning & Risk Scoring
# Each "# %%" block = one Jupyter cell. Paste in order.
# PREREQUISITE: Week 2 complete — model_ready.csv must exist.
# Install once if needed: pip install xgboost shap joblib
# ============================================================

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, URL
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)

sns.set_style("whitegrid")
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('model_ready.csv', index_col='application_id')
y = df['loan_default'].astype(int)
X = df.drop(columns=['loan_default'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Default rate — train {y_train.mean()*100:.1f}% / "
      f"test {y_test.mean()*100:.1f}%")

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'Logistic Regression': (LogisticRegression(
        class_weight='balanced', max_iter=1000,
        random_state=RANDOM_STATE), 'scaled'),
    'Decision Tree': (DecisionTreeClassifier(
        max_depth=5, class_weight='balanced',
        random_state=RANDOM_STATE), 'raw'),
    'Random Forest': (RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=20,
        class_weight='balanced_subsample', n_jobs=-1,
        random_state=RANDOM_STATE), 'raw'),
    'XGBoost': (XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric='logloss', n_jobs=-1,
        random_state=RANDOM_STATE), 'raw'),
}

In [ ]:
def evaluate_all(models_dict):
    rows, fitted = [], {}
    cv = StratifiedKFold(n_splits=5, shuffle=True,
                         random_state=RANDOM_STATE)
    for name, (model, mode) in models_dict.items():
        Xtr = X_train_s if mode == 'scaled' else X_train
        Xte = X_test_s if mode == 'scaled' else X_test

        t0 = time.time()
        cv_auc = cross_val_score(model, Xtr, y_train,
                                 cv=cv, scoring='roc_auc', n_jobs=-1)
        model.fit(Xtr, y_train)
        train_sec = time.time() - t0

        pred = model.predict(Xte)
        proba = model.predict_proba(Xte)[:, 1]
        rows.append({
            'Model': name,
            'CV_ROC_AUC': cv_auc.mean(),
            'Accuracy': accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred),
            'Recall': recall_score(y_test, pred),
            'F1': f1_score(y_test, pred),
            'ROC_AUC': roc_auc_score(y_test, proba),
            'Train_sec': round(train_sec, 1),
        })
        fitted[name] = (model, mode)
        print(f"{name:>20} | CV-AUC {cv_auc.mean():.3f} | "
              f"Recall {rows[-1]['Recall']:.3f} | "
              f"Test-AUC {rows[-1]['ROC_AUC']:.3f} | {train_sec:.0f}s")
    return pd.DataFrame(rows), fitted

results, fitted = evaluate_all(models)
results = results.sort_values('ROC_AUC', ascending=False)
print("\n===== MODEL COMPARISON (sorted by ROC-AUC) =====")
print(results.round(3).to_string(index=False))

In [ ]:
# DEFAULT RULE (change this if you want different behavior):
# Only models that clear the PRD's recall >= 0.70 target are eligible.
# Among eligible models, the one with the highest ROC-AUC wins.
# If NO model clears recall >= 0.70, we fall back to ranking all
# models by ROC-AUC and print a clear warning - don't let this
# fail silently.
results['passes_recall'] = results['Recall'] >= 0.70
eligible = results[results['passes_recall']]

if len(eligible) > 0:
    champion_row = eligible.sort_values('ROC_AUC', ascending=False).iloc[0]
    print(f"Champion chosen from {len(eligible)} model(s) meeting recall >= 0.70")
else:
    champion_row = results.sort_values('ROC_AUC', ascending=False).iloc[0]
    print("WARNING: no model met recall >= 0.70. Falling back to best "
          "ROC-AUC overall. Revisit threshold tuning in Cell 6, or "
          "reconsider feature engineering - this is below your PRD target.")

champion_name = champion_row['Model']
champion, champion_mode = fitted[champion_name]

print(f"\nCHAMPION: {champion_name}")
print(f"ROC-AUC: {champion_row['ROC_AUC']:.3f} "
      f"(PRD target >= 0.80: {'PASS' if champion_row['ROC_AUC'] >= 0.80 else 'BELOW TARGET'})")
print(f"Recall:  {champion_row['Recall']:.3f} "
      f"(PRD target >= 0.70: {'PASS' if champion_row['Recall'] >= 0.70 else 'BELOW TARGET'})")

# Confusion matrix for champion
Xte = X_test_s if champion_mode == 'scaled' else X_test
pred = champion.predict(Xte)
ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred),
    display_labels=['Repaid', 'Default']).plot(cmap='Blues',
                                               values_format='d')
plt.title(f'Champion ({champion_name}) — Confusion Matrix')
plt.show()

In [ ]:
# Default 0.5 threshold is rarely optimal for imbalanced risk data.
# Sweep thresholds and pick the one meeting recall >= 0.70 with the
# best precision — this is an explicit business trade-off.
proba_test = champion.predict_proba(Xte)[:, 1]

sweep = []
for thr in np.arange(0.20, 0.71, 0.05):
    p = (proba_test >= thr).astype(int)
    sweep.append({
        'threshold': round(thr, 2),
        'precision': precision_score(y_test, p),
        'recall': recall_score(y_test, p),
        'f1': f1_score(y_test, p),
    })
sweep_df = pd.DataFrame(sweep)
print(sweep_df.round(3).to_string(index=False))

sweep_df.set_index('threshold')[['precision', 'recall', 'f1']].plot(
    marker='o')
plt.title('Threshold Sweep — Precision / Recall / F1')
plt.ylabel('Score')
plt.axhline(0.70, color='red', linestyle='--', alpha=0.5,
            label='Recall target 0.70')
plt.legend()
plt.show()

# Pick threshold: lowest threshold where recall >= 0.70
valid = sweep_df[sweep_df['recall'] >= 0.70]
THRESHOLD = valid['threshold'].min() if len(valid) else 0.50
print(f"\nChosen decision threshold: {THRESHOLD:.2f}")

In [ ]:
import shap

Xtr = X_train_s if champion_mode == 'scaled' else X_train
Xtr_df = pd.DataFrame(Xtr, columns=X.columns)
Xte_df = pd.DataFrame(Xte, columns=X.columns)

explainer = shap.Explainer(champion, Xtr_df)
shap_values = explainer(Xte_df)

shap.plots.bar(shap_values, max_display=15)
shap.plots.beeswarm(shap_values, max_display=15)

# Ranked feature-importance table (PRD FR-08)
importance = pd.DataFrame({
    'feature': X.columns,
    'mean_abs_shap': np.abs(shap_values.values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)
importance.to_csv('feature_importance.csv', index=False)
print("\nTop 10 default drivers:")
print(importance.head(10).to_string(index=False))